# **Prepare images for the Cellpose command line (batch processing)**

This notebook loops over **one or more folders** of images, pulls out the channels Cellpose needs, and saves them as small **TIFF files**. It then writes the exact **Cellpose command-line (CLI) commands** needed to segment all of them in one go, using the pretrained `cyto3` model (cells) and `nuclei` model (nuclei).

The notebook does **not** run Cellpose itself, so it is quick. The segmentation is run afterwards from a terminal (see the last step).

### What you can choose
| Option | Meaning |
|---|---|
| Cell segmentation input | a **composite** of several channels (e.g. the cytoplasmic organelle channels) or a **single channel** (e.g. the plasma membrane) |
| Add nuclei to the cell segmentation | give `cyto3` the nuclear channel as a helper (a 2-channel image) or not (a 1-channel image) |
| Segment nuclei | run the `nuclei` model on the nuclear channel |

### Can the nuclei and the cells share one TIFF file?
**Yes, when the nuclei are used as a helper channel for the cells.** The prepared file then has two channels (channel 1 = cell signal, channel 2 = nuclei). `cyto3` uses both, and the `nuclei` model is simply told to read channel 2 only (`--chan 2 --chan2 0`). I tested this: it gives the same result as a separate nuclei-only file, and the same file is not stored twice.

**No, when the cells are segmented from the cell signal alone.** The cell file then has a single channel, so a separate nuclei-only file is saved for the `nuclei` model.

The notebook picks the right layout for you and tells you which one it used.

### The six steps
1. Settings (folders, channels, Cellpose parameters)
2. Find the images
3. Prepare the TIFF files (one per image)
4. Write the Cellpose commands
5. Run the commands in a terminal
6. Use the results

### What is created
Inside `OUTPUT_ROOT`, one sub-folder per source folder:

| Folder | Content |
|---|---|
| `cell_input\` | TIFF files for the cell segmentation (and nuclei too if the file is shared) |
| `nuclei_input\` | nuclei-only TIFF files (only when a separate file is needed) |
| `metadata\` | one `.json` file per image: original file, pixel size, downsampling, channel use |
| `cell_masks\`, `nuclei_masks\` | empty now; Cellpose writes the masks here |

Plus `run_cellpose_batch.bat`, which holds the commands.

---------------------
## **STEP 1: Settings**

#### &#x1F6D1; &#x270D; **User Input Required:**

This notebook now supports **two settings modes**:

1. `SETTINGS_SOURCE = "config"`  
   - Read `cellpose_config.json` from each dataset folder.
   - If missing, you can optionally fall back to manual inputs and auto-create the config file.

2. `SETTINGS_SOURCE = "manual"`  
   - Use the manual input variables (same format as the original notebook behavior).
   - For each dataset folder, the notebook writes `cellpose_config.json` (and optional history snapshot) with the settings used.

This guarantees each processed dataset folder ends with a config file that matches the parameters used for preparation and segmentation.

### Quick start

Use one of these three patterns:

1. **Config-first (recommended for production runs)**
   - Set `SETTINGS_SOURCE = "config"`
   - Set `STRICT_CONFIG = True`
   - Set `FALLBACK_TO_MANUAL_IF_MISSING_CONFIG = False`
   - Result: only datasets with `cellpose_config.json` are processed.

2. **Manual-first (recommended while tuning settings)**
   - Set `SETTINGS_SOURCE = "manual"`
   - Fill in the manual input fields in the next cell
   - Keep `WRITE_CONFIG_HISTORY = True`
   - Result: each processed dataset gets/updates `cellpose_config.json` plus a timestamped history snapshot.

3. **Hybrid (use configs when present, otherwise create from manual inputs)**
   - Set `SETTINGS_SOURCE = "config"`
   - Set `FALLBACK_TO_MANUAL_IF_MISSING_CONFIG = True`
   - Result: existing configs are used; missing configs are auto-created from manual inputs.

In [8]:
from pathlib import Path

#### USER INPUT REQUIRED ####
# Point to dataset folders (folder name should be the unique dataset ID).
SOURCE_FOLDERS = [
    Path(r"/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1"),
    # Path(r"/users/s/r/srhoads/MJFF_Cellpose-batch-testing/000001"),
]

# Where prepared TIFFs + command file are written.
OUTPUT_ROOT = Path(r"/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test")

# ---------- settings source ----------
SETTINGS_SOURCE = "config"   # "config" or "manual"
CONFIG_FILENAME = "cellpose_config.json"
STRICT_CONFIG = True          # only used when SETTINGS_SOURCE == "config"
FALLBACK_TO_MANUAL_IF_MISSING_CONFIG = True

# If config IM_TYPE has no files, auto-detect from common types in this order.
AUTO_DETECT_IM_TYPE_IF_EMPTY = True
AUTO_DETECT_IM_TYPE_CANDIDATES = [".tiff", ".tif", ".czi"]
UPDATE_CONFIG_IM_TYPE_ON_AUTODETECT = True

# ---------- config writing ----------
WRITE_CONFIG_HISTORY = True
HISTORY_DIRNAME = "cellpose_config_history"
CONFIG_SELECTED_BY = "prep_notebook"
CONFIG_NOTES = ""

SKIP_EXISTING = True

# ---------- manual settings (legacy-style inputs; used when SETTINGS_SOURCE == "manual") ----------
IM_TYPE = ".tiff"

SEGMENT_CELLS = True
SEGMENT_NUCLEI = True

# --- cell signal ---
CELL_SOURCE = "composite"      # "composite" or "single"
PM_CH = 7                      # plasma membrane channel
COMPOSITE_WEIGHTS = {0: 1,
                     1: 2,
                     2: 3,
                     3: 3,
                     4: 2,
                     5: 2,
                     6: 3}     # channel: weight (used if CELL_SOURCE = "composite")
INVERT_PM = False              # composite only: also add the inverted plasma membrane channel

# --- nuclei ---
NUC_CH = 1                     # nuclear stain channel
USE_NUCLEI_FOR_CELLS = True    # True: 2-channel cell file (cells + nuclei, shared with the nuclei model)

# --- image size ---
XY_DOWNSAMPLE = 1              # >1 shrinks XY (faster); masks are then at the shrunken size

# --- Cellpose parameters (diameters in pixels of the prepared image; 0 = estimate automatically) ---
CELL_DIAMETER = 500
NUC_DIAMETER = 150
FLOW_THRESHOLD = 0.2
CELLPROB_THRESHOLD = -3
MIN_SIZE_CELL = 5000
MIN_SIZE_NUC = 2000
BATCH_SIZE = 4
USE_GPU = True

# Optional custom models (None = pretrained cyto3/nuclei)
CP_CELL_MODEL_PATH = None
CP_NUC_MODEL_PATH = None

# Resource log upsert behavior from NEW_infer_masks_cellpose.ipynb
# (used for provenance in config files; default keeps one latest benchmark row per dataset)
PROFILE_UPSERT_KEYS = ["dataset_id"]

# Fallback defaults used to fill missing keys in configs
DEFAULT_SETTINGS = {
    "IM_TYPE": IM_TYPE,
    "SEGMENT_CELLS": SEGMENT_CELLS,
    "SEGMENT_NUCLEI": SEGMENT_NUCLEI,
    "CELL_SOURCE": CELL_SOURCE,
    "PM_CH": PM_CH,
    "COMPOSITE_WEIGHTS": COMPOSITE_WEIGHTS,
    "INVERT_PM": INVERT_PM,
    "NUC_CH": NUC_CH,
    "USE_NUCLEI_FOR_CELLS": USE_NUCLEI_FOR_CELLS,
    "XY_DOWNSAMPLE": XY_DOWNSAMPLE,
    "CELL_DIAMETER": CELL_DIAMETER,
    "NUC_DIAMETER": NUC_DIAMETER,
    "FLOW_THRESHOLD": FLOW_THRESHOLD,
    "CELLPROB_THRESHOLD": CELLPROB_THRESHOLD,
    "MIN_SIZE_CELL": MIN_SIZE_CELL,
    "MIN_SIZE_NUC": MIN_SIZE_NUC,
    "BATCH_SIZE": BATCH_SIZE,
    "USE_GPU": USE_GPU,
    "CP_CELL_MODEL_PATH": CP_CELL_MODEL_PATH,
    "CP_NUC_MODEL_PATH": CP_NUC_MODEL_PATH,
    "PROFILE_UPSERT_KEYS": PROFILE_UPSERT_KEYS,
}

#### &#x1F3C3; **Run code; no user input required**

Loads the tools used below and checks that your settings make sense. (If you see an error here, read the message: it says which setting to change.)

In [9]:
import json
from datetime import datetime
import numpy as np
import pandas as pd
import tifffile
from pathlib import Path
from skimage.transform import resize

from infer_subc.core.file_io import read_czi_image, list_image_files
from infer_subc.organelles.membrane import membrane_composite


_REQUIRED_CONFIG_KEYS = [
    "IM_TYPE", "SEGMENT_CELLS", "SEGMENT_NUCLEI", "CELL_SOURCE", "PM_CH", "NUC_CH",
    "USE_NUCLEI_FOR_CELLS", "XY_DOWNSAMPLE", "CELL_DIAMETER", "NUC_DIAMETER",
    "FLOW_THRESHOLD", "CELLPROB_THRESHOLD", "MIN_SIZE_CELL", "MIN_SIZE_NUC", "BATCH_SIZE", "USE_GPU",
]


def _canonical_settings_keys(raw):
    """Allow either flat keys or nested under a `settings` key."""
    if not isinstance(raw, dict):
        raise ValueError("Config content must be a JSON object.")
    if "settings" in raw and isinstance(raw["settings"], dict):
        merged = dict(raw["settings"])
        for k in ("dataset_id", "selected_on", "selected_by", "notes", "config_version"):
            if k in raw and k not in merged:
                merged[k] = raw[k]
        return merged
    return dict(raw)


def _manual_settings_from_inputs():
    return {
        "IM_TYPE": IM_TYPE,
        "SEGMENT_CELLS": SEGMENT_CELLS,
        "SEGMENT_NUCLEI": SEGMENT_NUCLEI,
        "CELL_SOURCE": CELL_SOURCE,
        "PM_CH": PM_CH,
        "COMPOSITE_WEIGHTS": COMPOSITE_WEIGHTS,
        "INVERT_PM": INVERT_PM,
        "NUC_CH": NUC_CH,
        "USE_NUCLEI_FOR_CELLS": USE_NUCLEI_FOR_CELLS,
        "XY_DOWNSAMPLE": XY_DOWNSAMPLE,
        "CELL_DIAMETER": CELL_DIAMETER,
        "NUC_DIAMETER": NUC_DIAMETER,
        "FLOW_THRESHOLD": FLOW_THRESHOLD,
        "CELLPROB_THRESHOLD": CELLPROB_THRESHOLD,
        "MIN_SIZE_CELL": MIN_SIZE_CELL,
        "MIN_SIZE_NUC": MIN_SIZE_NUC,
        "BATCH_SIZE": BATCH_SIZE,
        "USE_GPU": USE_GPU,
        "CP_CELL_MODEL_PATH": CP_CELL_MODEL_PATH,
        "CP_NUC_MODEL_PATH": CP_NUC_MODEL_PATH,
        "PROFILE_UPSERT_KEYS": PROFILE_UPSERT_KEYS,
    }


def _validate_and_normalize_settings(settings, source_label):
    merged = dict(DEFAULT_SETTINGS)
    merged.update(settings)

    if merged.get("CELL_SOURCE") not in ("composite", "single"):
        raise ValueError(f"{source_label}: CELL_SOURCE must be 'composite' or 'single'.")

    merged["SEGMENT_CELLS"] = bool(merged["SEGMENT_CELLS"])
    merged["SEGMENT_NUCLEI"] = bool(merged["SEGMENT_NUCLEI"])
    merged["USE_NUCLEI_FOR_CELLS"] = bool(merged["USE_NUCLEI_FOR_CELLS"])
    merged["USE_GPU"] = bool(merged.get("USE_GPU", True))

    merged["PM_CH"] = int(merged["PM_CH"])
    merged["NUC_CH"] = int(merged["NUC_CH"])
    merged["XY_DOWNSAMPLE"] = int(merged.get("XY_DOWNSAMPLE", 1))
    merged["CELL_DIAMETER"] = int(merged.get("CELL_DIAMETER", 0))
    merged["NUC_DIAMETER"] = int(merged.get("NUC_DIAMETER", 0))
    merged["MIN_SIZE_CELL"] = int(merged.get("MIN_SIZE_CELL", 0))
    merged["MIN_SIZE_NUC"] = int(merged.get("MIN_SIZE_NUC", 0))
    merged["BATCH_SIZE"] = int(merged.get("BATCH_SIZE", 4))

    merged["FLOW_THRESHOLD"] = float(merged.get("FLOW_THRESHOLD", 0.2))
    merged["CELLPROB_THRESHOLD"] = float(merged.get("CELLPROB_THRESHOLD", -3))

    # Keep profile upsert keys as list[str] for downstream provenance compatibility.
    puk = merged.get("PROFILE_UPSERT_KEYS", ["dataset_id"])
    if isinstance(puk, str):
        puk = [puk]
    if not isinstance(puk, (list, tuple)) or len(puk) == 0:
        puk = ["dataset_id"]
    merged["PROFILE_UPSERT_KEYS"] = [str(k) for k in puk]

    if merged["XY_DOWNSAMPLE"] < 1:
        raise ValueError(f"{source_label}: XY_DOWNSAMPLE must be >= 1.")

    if merged["CELL_SOURCE"] == "composite":
        cw = merged.get("COMPOSITE_WEIGHTS")
        if not isinstance(cw, dict) or not any(float(w) > 0 for w in cw.values()):
            raise ValueError(f"{source_label}: COMPOSITE_WEIGHTS must include at least one weight > 0.")

    missing = [k for k in _REQUIRED_CONFIG_KEYS if k not in merged]
    if missing:
        raise ValueError(f"{source_label}: missing required keys: {missing}")

    return merged


def _write_dataset_config(dataset_folder, settings, reason):
    dataset_folder = Path(dataset_folder)
    cfg_path = dataset_folder / CONFIG_FILENAME

    payload = {
        "dataset_id": dataset_folder.name,
        "config_version": "1.0",
        "selected_on": datetime.now().isoformat(timespec="seconds"),
        "selected_by": CONFIG_SELECTED_BY,
        "notes": CONFIG_NOTES,
        "generated_by": "NEW_cellpose-prep.ipynb",
        "generation_reason": reason,
        "settings": settings,
    }

    text = json.dumps(payload, indent=2)
    cfg_path.write_text(text)

    if WRITE_CONFIG_HISTORY:
        hist_dir = dataset_folder / HISTORY_DIRNAME
        hist_dir.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        hist_path = hist_dir / f"cellpose_config_{stamp}.json"
        hist_path.write_text(text)

    return cfg_path


def _update_config_im_type(cfg_path, im_type_value):
    """Persist auto-detected IM_TYPE back into config file to keep config and data aligned."""
    try:
        raw = json.loads(Path(cfg_path).read_text())
        if "settings" in raw and isinstance(raw["settings"], dict):
            raw["settings"]["IM_TYPE"] = im_type_value
        else:
            raw["IM_TYPE"] = im_type_value
        Path(cfg_path).write_text(json.dumps(raw, indent=2))
    except Exception as e:
        print(f"WARNING: failed to update config IM_TYPE at {cfg_path}: {type(e).__name__}: {e}")


def _load_or_make_dataset_config(dataset_folder):
    dataset_folder = Path(dataset_folder)
    mode = str(SETTINGS_SOURCE).lower().strip()
    cfg_path = dataset_folder / CONFIG_FILENAME

    if mode == "manual":
        settings = _validate_and_normalize_settings(_manual_settings_from_inputs(), "manual inputs")
        cfg_path = _write_dataset_config(dataset_folder, settings, reason="manual-settings")
        return settings, cfg_path, "manual"

    if mode != "config":
        raise ValueError("SETTINGS_SOURCE must be 'config' or 'manual'.")

    if cfg_path.exists():
        loaded = _canonical_settings_keys(json.loads(cfg_path.read_text()))
        settings = _validate_and_normalize_settings(loaded, str(cfg_path))
        return settings, cfg_path, "config-file"

    if FALLBACK_TO_MANUAL_IF_MISSING_CONFIG:
        settings = _validate_and_normalize_settings(_manual_settings_from_inputs(), f"manual fallback for {dataset_folder}")
        cfg_path = _write_dataset_config(dataset_folder, settings, reason="missing-config-manual-fallback")
        return settings, cfg_path, "manual-fallback"

    if STRICT_CONFIG:
        raise FileNotFoundError(f"Missing {CONFIG_FILENAME} in {dataset_folder}")

    settings = _validate_and_normalize_settings(DEFAULT_SETTINGS, f"default fallback for {dataset_folder}")
    cfg_path = _write_dataset_config(dataset_folder, settings, reason="missing-config-default-fallback")
    return settings, cfg_path, "default-fallback"


if not isinstance(SOURCE_FOLDERS, (list, tuple)):
    raise ValueError("SOURCE_FOLDERS must be a list, e.g. [Path('/path/to/dataset')].")

print(f"Settings source: {SETTINGS_SOURCE}")
print(f"Config filename: {CONFIG_FILENAME}")
print(f"Fallback to manual when config missing: {FALLBACK_TO_MANUAL_IF_MISSING_CONFIG}")

Settings source: config
Config filename: cellpose_config.json
Fallback to manual when config missing: True


---------------------
## **STEP 2: Find the images**

#### &#x1F3C3; **Run code; no user input required**

Lists the images found in each source folder. Each folder gets its own sub-folder inside `OUTPUT_ROOT`; if two folders have the same name, a number is added so nothing is overwritten.

In [10]:
datasets, used_names = [], set()
for folder in SOURCE_FOLDERS:
    folder = Path(folder)
    if not folder.exists():
        print(f"WARNING: folder not found, skipped: {folder}")
        continue

    try:
        cfg, cfg_path, cfg_source = _load_or_make_dataset_config(folder)
    except Exception as e:
        print(f"WARNING: config error for {folder}: {type(e).__name__}: {e}")
        continue

    ds_im_type = str(cfg.get("IM_TYPE", DEFAULT_SETTINGS["IM_TYPE"]))
    files = list_image_files(folder, ds_im_type)

    # Optional auto-detect fallback if config IM_TYPE does not match files in folder.
    if (not files) and AUTO_DETECT_IM_TYPE_IF_EMPTY:
        detected = None
        for cand in AUTO_DETECT_IM_TYPE_CANDIDATES:
            cand = str(cand)
            cand_files = list_image_files(folder, cand)
            if cand_files:
                detected = cand
                files = cand_files
                break

        if detected is not None:
            print(f"WARNING: no '{ds_im_type}' files in {folder}. Auto-detected '{detected}' ({len(files)} files).")
            ds_im_type = detected
            cfg["IM_TYPE"] = detected
            if UPDATE_CONFIG_IM_TYPE_ON_AUTODETECT and cfg_path is not None:
                _update_config_im_type(cfg_path, detected)
                print(f"Updated config IM_TYPE in {cfg_path} -> '{detected}'")

    if not files:
        print(f"WARNING: no '{ds_im_type}' files found in {folder}")
        continue

    name = base = folder.name or "images"
    n = 2
    while name in used_names:
        name, n = f"{base}_{n}", n + 1
    used_names.add(name)

    datasets.append({
        "name": name,
        "folder": folder,
        "files": files,
        "config": cfg,
        "config_path": cfg_path,
        "config_source": cfg_source,
        "im_type": ds_im_type,
    })

if not datasets:
    raise FileNotFoundError("No usable datasets found. Check SOURCE_FOLDERS, IM_TYPE, and dataset config files/settings.")

pd.set_option("display.max_colwidth", None)
pd.DataFrame([
    {
        "dataset": d["name"],
        "source folder": str(d["folder"]),
        "images": len(d["files"]),
        "im_type": d["im_type"],
        "config": str(d["config_path"]),
        "config_source": d["config_source"],
        "cell_source": d["config"]["CELL_SOURCE"],
        "segment_cells": d["config"]["SEGMENT_CELLS"],
        "segment_nuclei": d["config"]["SEGMENT_NUCLEI"],
    }
    for d in datasets
])

Updated config IM_TYPE in /users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/cellpose_config.json -> '.tiff'


,dataset,source folder,images,im_type,config,config_source,cell_source,segment_cells,segment_nuclei
0,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1,10,.tiff,/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/cellpose_config.json,config-file,composite,True,True


---------------------
## **STEP 3: Prepare the TIFF files**

#### &#x1F3C3; **Run code; no user input required**

For every image: read it, build the cell signal (composite or single channel) and/or the nuclei channel, scale each to the range 0-1 (1st to 99.8th percentile), shrink in XY if requested, and save as a TIFF. A small `.json` file with the pixel size and settings is saved next to it.

This can take a little while for large images. If one image fails, the error is recorded and the rest continue.

In [ ]:
def _norm01(v):
    v = np.asarray(v, dtype=np.float32)
    p1, p99 = np.percentile(v, (1, 99.8))
    return np.clip((v - p1) / (p99 - p1 + 1e-8), 0, 1).astype(np.float32)


def _shrink(vol, xy_downsample):
    if xy_downsample == 1:
        return vol
    shape = (vol.shape[0], round(vol.shape[1] / xy_downsample), round(vol.shape[2] / xy_downsample))
    return resize(vol, shape, order=1, anti_aliasing=True).astype(np.float32)


def _cell_signal(img, cfg):
    """Cell signal as ZYX, from a composite of channels or a single channel."""
    if cfg["CELL_SOURCE"] == "single":
        return _norm01(img[int(cfg["PM_CH"])])

    weights = {f"weight_ch{int(c)}": float(w) for c, w in cfg["COMPOSITE_WEIGHTS"].items() if float(w) > 0}
    comp = membrane_composite(
        img,
        Invert_PM=bool(cfg.get("INVERT_PM", False)),
        PM_Channel=int(cfg["PM_CH"]),
        **weights,
    )
    return _norm01(comp)


records = []
for ds in datasets:
    cfg = ds["config"]

    segment_cells = bool(cfg["SEGMENT_CELLS"])
    segment_nuclei = bool(cfg["SEGMENT_NUCLEI"])
    use_nuc_for_cells = bool(cfg["USE_NUCLEI_FOR_CELLS"])
    cell_file_has_nuc = bool(segment_cells and use_nuc_for_cells)
    separate_nuc_file = bool(segment_nuclei and not cell_file_has_nuc)
    need_nuc_channel = bool(cell_file_has_nuc or segment_nuclei)

    xy_downsample = int(cfg["XY_DOWNSAMPLE"])
    pm_ch = int(cfg["PM_CH"])
    nuc_ch = int(cfg["NUC_CH"])

    needed_channels = [nuc_ch] if need_nuc_channel else []
    if segment_cells:
        if cfg["CELL_SOURCE"] == "composite":
            needed_channels += [int(c) for c, w in cfg["COMPOSITE_WEIGHTS"].items() if float(w) > 0]
            if bool(cfg.get("INVERT_PM", False)):
                needed_channels.append(pm_ch)
        else:
            needed_channels.append(pm_ch)

    root = Path(OUTPUT_ROOT) / ds["name"]
    dirs = {k: root / k for k in ("cell_input", "nuclei_input", "metadata", "cell_masks", "nuclei_masks")}
    ds["dirs"] = dirs
    ds["runtime"] = {
        "cell_file_has_nuc": cell_file_has_nuc,
        "separate_nuc_file": separate_nuc_file,
        "need_nuc_channel": need_nuc_channel,
    }

    for k, d in dirs.items():
        if (k == "cell_input" and segment_cells) or (k == "nuclei_input" and separate_nuc_file) \
           or (k == "cell_masks" and segment_cells) or (k == "nuclei_masks" and segment_nuclei) or k == "metadata":
            d.mkdir(parents=True, exist_ok=True)

    for src in ds["files"]:
        stem = Path(src).stem
        cell_f = dirs["cell_input"] / f"{stem}.tif"
        nuc_f = dirs["nuclei_input"] / f"{stem}.tif"
        meta_f = dirs["metadata"] / f"{stem}.json"

        expected = [meta_f] + ([cell_f] if segment_cells else []) + ([nuc_f] if separate_nuc_file else [])

        rec = {
            "dataset": ds["name"],
            "image": stem,
            "status": "",
            "anisotropy": np.nan,
            "prepared shape (Z,Y,X)": "",
            "message": "",
        }

        try:
            if SKIP_EXISTING and all(f.exists() for f in expected):
                info = json.loads(meta_f.read_text())
                rec.update(
                    status="skipped (already prepared)",
                    anisotropy=info.get("anisotropy", np.nan),
                    **{"prepared shape (Z,Y,X)": str(tuple(info.get("prepared_shape_zyx", [])))},
                )
            else:
                img, meta = read_czi_image(src)
                bad = [c for c in sorted(set(needed_channels)) if c >= img.shape[0]]
                if bad:
                    raise ValueError(f"image has {img.shape[0]} channels (0-{img.shape[0] - 1}) but channel(s) {bad} are needed")

                scale = [float(s) for s in meta["scale"]]
                aniso = scale[0] / (scale[1] * xy_downsample)

                nuc = _shrink(_norm01(img[nuc_ch]), xy_downsample) if need_nuc_channel else None
                cell = _shrink(_cell_signal(img, cfg), xy_downsample) if segment_cells else None

                if segment_cells:
                    out = np.stack([cell, nuc], axis=1) if cell_file_has_nuc else cell  # ZCYX or ZYX
                    tifffile.imwrite(cell_f, out.astype(np.float32))
                if separate_nuc_file:
                    tifffile.imwrite(nuc_f, nuc.astype(np.float32))

                shape = (cell if cell is not None else nuc).shape
                meta_f.write_text(json.dumps({
                    "source_file": str(src),
                    "scale_zyx": scale,
                    "xy_downsample": xy_downsample,
                    "anisotropy": aniso,
                    "original_shape_zyx": list(img.shape[1:]),
                    "prepared_shape_zyx": list(shape),
                    "cell_file_channels": ["cell signal", "nuclei"] if cell_file_has_nuc else (["cell signal"] if segment_cells else []),
                    "cell_source": cfg["CELL_SOURCE"] if segment_cells else None,
                    "nuclei_channel": nuc_ch if need_nuc_channel else None,
                    "pm_channel": pm_ch,
                    "composite_weights": cfg.get("COMPOSITE_WEIGHTS") if cfg["CELL_SOURCE"] == "composite" else None,
                    "invert_pm": bool(cfg.get("INVERT_PM", False)),
                    "config_file": str(ds["config_path"]) if ds.get("config_path") else None,
                    "config_version": cfg.get("config_version"),
                }, indent=1))

                rec.update(status="prepared", anisotropy=round(aniso, 4), **{"prepared shape (Z,Y,X)": str(tuple(shape))})
                del img, nuc, cell

        except Exception as e:
            rec.update(status="ERROR", message=f"{type(e).__name__}: {e}")

        records.append(rec)
        print(f"{ds['name']} / {stem}: {rec['status']} {rec['message']}")

prep_table = pd.DataFrame(records)
prep_table.to_csv(Path(OUTPUT_ROOT) / "prepared_images.csv", index=False)
if (prep_table["status"] == "ERROR").any():
    print("\nSome images failed (see the 'message' column below). The others were prepared normally.")
prep_table

---------------------
## **STEP 4: Write the Cellpose commands**

#### &#x1F3C3; **Run code; no user input required**

Builds the Cellpose command(s) for every folder and saves them in `run_cellpose_batch.bat` inside `OUTPUT_ROOT`.

* The cell command runs `cyto3` (`--chan 1 --chan2 2` when the file has nuclei, `--chan 0 --chan2 0` for a single channel).
* The nuclei command runs `nuclei`. With a shared file it reads channel 2 only (`--chan 2 --chan2 0`).
* The pixel-size ratio between Z and XY (`--anisotropy`) comes from each image's metadata. If images in a folder have different values, a separate command is written per image.

In [ ]:
def _cli(
    indir_or_file,
    savedir,
    model,
    chan,
    chan2,
    diameter,
    min_size,
    aniso,
    flow_threshold,
    cellprob_threshold,
    batch_size,
    use_gpu,
    channel_axis=None,
    is_file=False,
):
    parts = [
        "python -m cellpose",
        f'--image_path "{indir_or_file}"' if is_file else f'--dir "{indir_or_file}"',
        f'--pretrained_model "{model}"' if str(model).startswith("/") else f"--pretrained_model {model}",
        "--do_3D",
        "--z_axis 0",
    ]
    if channel_axis is not None:
        parts.append(f"--channel_axis {channel_axis}")

    parts += [
        f"--chan {chan}",
        f"--chan2 {chan2}",
        f"--diameter {diameter}",
        f"--anisotropy {aniso:.4f}",
        f"--flow_threshold {flow_threshold}",
        f"--cellprob_threshold {cellprob_threshold}",
        f"--min_size {min_size}",
        f"--batch_size {batch_size}",
    ]
    if use_gpu:
        parts.append("--use_gpu")
    parts += ["--save_tif", "--no_npy", f'--savedir "{savedir}"', "--verbose"]
    return " ".join(parts)


good = prep_table[prep_table["status"].isin(["prepared", "skipped (already prepared)"])]
if good.empty:
    raise RuntimeError("No images were prepared successfully, so there are no commands to write. See the table above.")

lines = [
    "@echo off",
    "chcp 65001 >nul",
    "REM Run this file from a terminal where the environment with Cellpose 3 (cellpose<4) is activated.",
    "REM The masks are saved as <image name>_cp_masks.tif in the cell_masks / nuclei_masks folders.",
    "",
]

for ds in datasets:
    sub = good[good["dataset"] == ds["name"]]
    if sub.empty:
        continue

    cfg = ds["config"]
    rt = ds["runtime"]
    dirs = ds["dirs"]

    segment_cells = bool(cfg["SEGMENT_CELLS"])
    segment_nuclei = bool(cfg["SEGMENT_NUCLEI"])
    cell_file_has_nuc = bool(rt["cell_file_has_nuc"])
    separate_nuc_file = bool(rt["separate_nuc_file"])

    flow_threshold = cfg["FLOW_THRESHOLD"]
    cellprob_threshold = cfg["CELLPROB_THRESHOLD"]
    batch_size = cfg["BATCH_SIZE"]
    use_gpu = bool(cfg.get("USE_GPU", True))

    cell_model = cfg.get("CP_CELL_MODEL_PATH") or "cyto3"
    nuc_model = cfg.get("CP_NUC_MODEL_PATH") or "nuclei"

    groups = sub.groupby(sub["anisotropy"].round(3))

    lines.append(f"REM ===== {ds['name']} =====")
    lines.append(f"REM config: {ds['config_path']}")

    for name, wanted in (("cells", segment_cells), ("nuclei", segment_nuclei)):
        if not wanted:
            continue

        if name == "cells":
            indir, savedir, model = dirs["cell_input"], dirs["cell_masks"], cell_model
            chan, chan2, axis = (1, 2, 1) if cell_file_has_nuc else (0, 0, None)
            diam, msize = cfg["CELL_DIAMETER"], cfg["MIN_SIZE_CELL"]
        else:
            savedir, model, diam, msize = dirs["nuclei_masks"], nuc_model, cfg["NUC_DIAMETER"], cfg["MIN_SIZE_NUC"]
            if separate_nuc_file:
                indir, chan, chan2, axis = dirs["nuclei_input"], 0, 0, None
            else:
                indir, chan, chan2, axis = dirs["cell_input"], 2, 0, 1

        if len(groups) == 1:
            aniso = float(sub["anisotropy"].iloc[0])
            lines.append(f"REM {name}")
            lines.append(_cli(
                indir, savedir, model, chan, chan2, diam, msize, aniso,
                flow_threshold, cellprob_threshold, batch_size, use_gpu,
                channel_axis=axis,
            ))
        else:
            lines.append(f"REM {name}: images have different Z/XY pixel-size ratios, so one command per image")
            for _, r in sub.iterrows():
                lines.append(_cli(
                    Path(indir) / f"{r['image']}.tif", savedir, model, chan, chan2, diam, msize,
                    float(r["anisotropy"]), flow_threshold, cellprob_threshold, batch_size, use_gpu,
                    channel_axis=axis, is_file=True,
                ))
    lines.append("")

lines.append("echo Done.")

bat_file = Path(OUTPUT_ROOT) / "run_cellpose_batch.bat"
bat_file.write_text("\r\n".join(lines) + "\r\n", encoding="utf-8")
print(f"Commands saved to:\n{bat_file}\n")
print("\n".join(lines))

---------------------
## **STEP 5: Run Cellpose from a terminal**

1. Open **Anaconda Prompt** (or any terminal).
2. Activate an environment that has **Cellpose 3** (`cellpose<4`; Cellpose 4 has no `cyto3`/`nuclei` models). For example: `conda activate waveorder-cytoland`.
3. Run the file printed above, e.g.
   `"C:\...\cellpose_batch\run_cellpose_batch.bat"`
   (or copy single lines from the printed commands and run them one at a time).

Notes:
* The **first time** Cellpose runs it uses the model files it already has. If it cannot find `cyto3` or `nuclei`, it will try to download them.
* Progress is shown in the terminal. Re-running the file overwrites existing masks.
* Output folders must already exist; this notebook creates them, so do not delete them.

---------------------
## **STEP 6: Use the results**

* Masks are saved as `<image name>_cp_masks.tif` (labels: 0 = background, 1, 2, 3, ... = objects) in `cell_masks\` and `nuclei_masks\`.
* If `XY_DOWNSAMPLE` was above 1, the masks are smaller than the original image. The `metadata\<image name>.json` file stores the factor and the original shape so they can be scaled back up.
* `prepared_images.csv` in `OUTPUT_ROOT` lists every image with its status.
* To match each nucleus to a cell, or to open masks next to the image in Napari, use the same approach as in `NEW_infer_masks_cellpose.ipynb`.

#### Troubleshooting
| Problem | What to do |
|---|---|
| "No images found" | Check `SOURCE_FOLDERS` (raw strings: `Path(r"C:\...")`) and `IM_TYPE`. |
| "image has N channels but channel(s) [..] are needed" | A channel number in the settings does not exist in that image. |
| `FileExistsError: --savedir ... does not exist` | A `cell_masks`/`nuclei_masks` folder was deleted. Re-run Step 3. |
| Cellpose reports it cannot find `cyto3` or `nuclei` | You are in a Cellpose 4 environment. Use one with `cellpose<4`. |
| Out of GPU memory | Lower `BATCH_SIZE` or raise `XY_DOWNSAMPLE`, then re-run Steps 3-4 with `SKIP_EXISTING = False`. |
| Changed a setting that affects the images (channels, weights, downsampling) | Set `SKIP_EXISTING = False` so the files are rebuilt. |